# ANVESH: Model 1 — Phishing Baseline Classifier (Colab Training)
## SIH26106: Intelligent Email Forensics, BEC Detection & Attribution Framework

> **Important Forensic & Governance Notice**:
> *"This model is an evidence layer and must not be used as an autonomous final threat verdict."*
>
> **Model 1 Architecture**: Pure text classifier (`Subject` + `Body` -> TF-IDF -> Logistic Regression).
> - **Target**: Binary (`BENIGN` = 0 vs `THREAT_PHISHING` = 1).
> - **Exclusions**: `THREAT_ADVANCE_FEE` (419 scams) are excluded from Model 1 binary training.
> - **No Header / Verdict Contamination**: Zero access to SPF/DKIM/DMARC, IPs, DNS, RDAP, VirusTotal, AbuseIPDB, or risk scores.
> - **Governance Invariants**: IWSPA-AP Independent Test Set and ANVESH Challenge Benchmark are **STRICTLY NOT USED**.

### Section 1 — Environment Setup & Reproducibility

In [ ]:
import os
import sys
import json
import html
import re
import unicodedata
import datetime
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    precision_recall_curve, roc_curve
)
from sklearn.pipeline import Pipeline

# Fix global seed for exact reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Scikit-Learn Version: {sklearn.__version__}")
print(f"Joblib Version: {joblib.__version__}")
print("Environment initialized successfully.")

### Section 2 — Upload Development Datasets
Upload `train.jsonl` and `val.jsonl` (or load them automatically if cloned locally).

In [ ]:
# Check if running in Google Colab or local repo
IN_COLAB = 'google.colab' in sys.modules

train_file = Path("train.jsonl")
val_file = Path("val.jsonl")

if not train_file.exists() or not val_file.exists():
    # Check common relative repo paths
    candidates = [
        Path("ml/datasets/processed"),
        Path("../ml/datasets/processed"),
        Path("datasets/processed"),
        Path("../datasets/processed")
    ]
    found = False
    for c in candidates:
        if (c / "train.jsonl").exists() and (c / "val.jsonl").exists():
            train_file = c / "train.jsonl"
            val_file = c / "val.jsonl"
            found = True
            print(f"Found datasets at: {c}")
            break
    
    if not found and IN_COLAB:
        print("Please upload train.jsonl and val.jsonl using the upload prompt below:")
        from google.colab import files
        uploaded = files.upload()
        train_file = Path("train.jsonl")
        val_file = Path("val.jsonl")

print(f"Using train dataset path: {train_file.resolve()}")
print(f"Using val dataset path:   {val_file.resolve()}")

### Section 3 — Dataset Loading & Verification

In [ ]:
def load_jsonl(filepath):
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

train_raw = load_jsonl(train_file)
val_raw = load_jsonl(val_file)

print(f"Total Raw Training Records:   {len(train_raw):,}")
print(f"Total Raw Validation Records: {len(val_raw):,}")

### Section 4 — Label Filtering (Model 1 Binary Target)
We strictly filter for `BENIGN` ($y=0$) and `THREAT_PHISHING` ($y=1$).  
`THREAT_ADVANCE_FEE` records are explicitly excluded from Model 1 training.

In [ ]:
train_binary = [r for r in train_raw if r.get("anvesh_label") in ("BENIGN", "THREAT_PHISHING")]
val_binary = [r for r in val_raw if r.get("anvesh_label") in ("BENIGN", "THREAT_PHISHING")]

train_excluded_419 = [r for r in train_raw if r.get("anvesh_label") == "THREAT_ADVANCE_FEE"]
val_excluded_419 = [r for r in val_raw if r.get("anvesh_label") == "THREAT_ADVANCE_FEE"]

print("=" * 60)
print("MODEL 1 BINARY DATASET COMPOSITION")
print("=" * 60)
print(f"Training Sample Count (Binary):   {len(train_binary):,}")
print(f"  - THREAT_PHISHING Count:        {sum(1 for r in train_binary if r['anvesh_label'] == 'THREAT_PHISHING'):,}")
print(f"  - BENIGN Count:                 {sum(1 for r in train_binary if r['anvesh_label'] == 'BENIGN'):,}")
print(f"  - Excluded THREAT_ADVANCE_FEE:  {len(train_excluded_419):,}")

print(f"\nValidation Sample Count (Binary): {len(val_binary):,}")
print(f"  - THREAT_PHISHING Count:        {sum(1 for r in val_binary if r['anvesh_label'] == 'THREAT_PHISHING'):,}")
print(f"  - BENIGN Count:                 {sum(1 for r in val_binary if r['anvesh_label'] == 'BENIGN'):,}")
print(f"  - Excluded THREAT_ADVANCE_FEE:  {len(val_excluded_419):,}")
print("=" * 60)

### Section 5 — Deterministic Text Preprocessing
Implements self-contained Unicode NFKC normalization, HTML tag stripping, security tokenization (`__URL_TOKEN__`, `__EMAIL_TOKEN__`, `__CURRENCY_TOKEN__`), and whitespace normalization.

In [ ]:
URL_PATTERN = re.compile(r"""(?:https?://|www\.)[^\s<>"'{}|\^`]+""", re.IGNORECASE)
EMAIL_PATTERN = re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b')
CURRENCY_PATTERN = re.compile(r'[\$£€₹¥]\s*\d+(?:[.,]\d+)*(?:\s*(?:million|billion|thousand|k|m|usd|inr|eur|gbp))?', re.IGNORECASE)
HTML_TAG_PATTERN = re.compile(r'<[^>]+>')
MIME_ARTIFACT_PATTERN = re.compile(r'(?:--=_[A-Za-z0-9._=-]+|Content-Type:[^\n]+|charset=[^\n]+|Content-Transfer-Encoding:[^\n]+)', re.IGNORECASE)
WHITESPACE_PATTERN = re.compile(r'\s+')

def clean_email_text(text):
    if text is None or not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = html.unescape(text)
    text = MIME_ARTIFACT_PATTERN.sub(" ", text)
    text = HTML_TAG_PATTERN.sub(" ", text)
    text = URL_PATTERN.sub(" __URL_TOKEN__ ", text)
    text = EMAIL_PATTERN.sub(" __EMAIL_TOKEN__ ", text)
    text = CURRENCY_PATTERN.sub(" __CURRENCY_TOKEN__ ", text)
    return WHITESPACE_PATTERN.sub(" ", text).strip()

def normalize_email_pair(subject, body):
    s = clean_email_text(subject)
    b = clean_email_text(body)
    return f"{s} {b}" if (s and b) else (s or b or "empty_email_content")

print("Preprocessing functions initialized.")

### Section 6 — Train/Validation Preparation

In [ ]:
X_train = [normalize_email_pair(r.get("subject", ""), r.get("body", "")) for r in train_binary]
y_train = np.array([1 if r["anvesh_label"] == "THREAT_PHISHING" else 0 for r in train_binary], dtype=int)

X_val = [normalize_email_pair(r.get("subject", ""), r.get("body", "")) for r in val_binary]
y_val = np.array([1 if r["anvesh_label"] == "THREAT_PHISHING" else 0 for r in val_binary], dtype=int)

print(f"Prepared X_train: {len(X_train):,} texts, y_train: {len(y_train):,} labels")
print(f"Prepared X_val:   {len(X_val):,} texts, y_val:   {len(y_val):,} labels")

### Section 7 — Model 1 Training (TF-IDF + Logistic Regression)

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    sublinear_tf=True,
    token_pattern=r"(?u)\b\w+\b|__\w+__"
)

lr_classifier = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    max_iter=1000,
    solver="lbfgs"
)

pipeline = Pipeline([
    ("tfidf", tfidf_vectorizer),
    ("clf", lr_classifier)
])

print("Training Model 1 baseline on training corpus (6,171 records)...")
start_time = datetime.datetime.now()
pipeline.fit(X_train, y_train)
elapsed = (datetime.datetime.now() - start_time).total_seconds()
print(f"Training complete in {elapsed:.2f} seconds.")

### Section 8 — Validation Metrics

> **Governance Label**:
> **"Development validation — NOT independent test performance"**

In [ ]:
val_proba = pipeline.predict_proba(X_val)
val_phishing_proba = val_proba[:, 1]
val_pred = (val_phishing_proba >= 0.5).astype(int)

acc = accuracy_score(y_val, val_pred)
prec = precision_score(y_val, val_pred, pos_label=1)
rec = recall_score(y_val, val_pred, pos_label=1)
f1 = f1_score(y_val, val_pred, pos_label=1)
roc_auc = roc_auc_score(y_val, val_phishing_proba)

print("=" * 65)
print("Development validation — NOT independent test performance")
print("=" * 65)
print(f"  Accuracy:   {acc:.4f} ({acc*100:.2f}%)")
print(f"  Precision:  {prec:.4f} ({prec*100:.2f}%)")
print(f"  Recall:     {rec:.4f} ({rec*100:.2f}%)")
print(f"  F1-Score:   {f1:.4f} ({f1*100:.2f}%)")
print(f"  ROC-AUC:    {roc_auc:.4f}")
print("=" * 65)

### Section 9 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_val, val_pred)
tn, fp, fn, tp = [int(x) for x in cm.ravel()]

print("Confusion Matrix (Development Validation):")
print(f"  True Positives (TP):  {tp}")
print(f"  True Negatives (TN):  {tn}")
print(f"  False Positives (FP): {fp}")
print(f"  False Negatives (FN): {fn}")
print(f"  Total Evaluated:      {tp + tn + fp + fn}")

### Section 10 — Probability Distribution Analysis

In [ ]:
benign_p = val_phishing_proba[y_val == 0]
phish_p = val_phishing_proba[y_val == 1]

print("Probability Distribution Summary:")
print(f"  Overall Min:    {np.min(val_phishing_proba):.4f}")
print(f"  Overall Max:    {np.max(val_phishing_proba):.4f}")
print(f"  Overall Median: {np.median(val_phishing_proba):.4f}")
print(f"  Benign Subset Median:   {np.median(benign_p):.4f} (Mean: {np.mean(benign_p):.4f})")
print(f"  Phishing Subset Median: {np.median(phish_p):.4f} (Mean: {np.mean(phish_p):.4f})")

bins = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.00001]
b_hist, _ = np.histogram(benign_p, bins=bins)
p_hist, _ = np.histogram(phish_p, bins=bins)
bin_labels = ['0.00-0.10', '0.10-0.20', '0.20-0.30', '0.30-0.40', '0.40-0.50', '0.50-0.60', '0.60-0.70', '0.70-0.80', '0.80-0.90', '0.90-1.00']

print("\nHistogram of Predicted Phishing Probabilities:")
for lbl, bc, pc in zip(bin_labels, b_hist, p_hist):
    print(f"  {lbl}: Actual Benign = {bc:<5} | Actual Phishing = {pc:<5}")

### Section 11 — Top Statistical TF-IDF Features

In [ ]:
feature_names = tfidf_vectorizer.get_feature_names_out()
coefs = lr_classifier.coef_[0]

top_pos_idx = np.argsort(coefs)[-15:][::-1]
top_neg_idx = np.argsort(coefs)[:15]

print("Top 15 Statistical Features Associated with THREAT_PHISHING:")
for idx in top_pos_idx:
    print(f"  {feature_names[idx]:<25} (weight: +{coefs[idx]:.4f})")

print("\nTop 15 Statistical Features Associated with BENIGN:")
for idx in top_neg_idx:
    print(f"  {feature_names[idx]:<25} (weight: {coefs[idx]:.4f})")

### Section 12 — Save Model Artifacts

In [ ]:
output_dir = Path("models/phishing_baseline_v1")
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "model.joblib"
metadata_path = output_dir / "metadata.json"
top_features_path = output_dir / "top_features.json"

joblib.dump(pipeline, model_path)

top_features_data = {
    "top_positive_features_phishing": [
        {"feature": str(feature_names[i]), "weight": round(float(coefs[i]), 4), "association": "THREAT_PHISHING"}
        for i in np.argsort(coefs)[-30:][::-1]
    ],
    "top_negative_features_benign": [
        {"feature": str(feature_names[i]), "weight": round(float(coefs[i]), 4), "association": "BENIGN"}
        for i in np.argsort(coefs)[:30]
    ]
}
with open(top_features_path, "w", encoding="utf-8") as f:
    json.dump(top_features_data, f, indent=2)

metadata_data = {
    "model_name": "anvesh_phishing_baseline",
    "model_version": "1.0.0",
    "training_date": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "random_seed": RANDOM_SEED,
    "development_validation_metrics": {
        "accuracy": round(float(acc), 4),
        "precision": round(float(prec), 4),
        "recall": round(float(rec), 4),
        "f1_score": round(float(f1), 4),
        "roc_auc": round(float(roc_auc), 4)
    },
    "governance_notice": "This model is an evidence layer and must not be used as an autonomous final threat verdict."
}
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata_data, f, indent=2)

print(f"Artifacts successfully saved to {output_dir.resolve()}")

### Section 13 — Download Model Artifacts (Colab Only)

In [ ]:
if IN_COLAB:
    from google.colab import files
    print("Downloading model artifacts...")
    files.download(str(model_path))
    files.download(str(metadata_path))
    files.download(str(top_features_path))
else:
    print(f"Artifacts ready locally at: {output_dir.resolve()}")

### Section 14 — Governance Invariants Verification

```
IWSPA independent test set: NOT USED
ANVESH challenge benchmark: NOT USED
API keys / Secrets:         NONE INCLUDED
```

This concludes the self-contained Google Colab training run.